# Neural Ledger — Coding Agent Failure Memory

This notebook walks through the canonical proof scenario:

> A coding agent previously encountered a GitHub API 401 failure caused by an expired token.
> Later, facing a similar error, it should recall the true cause — ranked above misleading near-matches.
> Positive feedback then reinforces that recall for future encounters.

**Arc:** `attempt → failure → remember cause → recall → feedback → better ranking`

## Setup

In [1]:
import sys
sys.path.insert(0, str(__import__("pathlib").Path("..").resolve()))
import json
from pathlib import Path
from neural_ledger import Memory

DATASET = Path('..') / 'proof' / 'datasets' / 'coding_agent_failure_memory.json'
data = json.loads(DATASET.read_text())
print(f"Loaded {len(data['records'])} records, {len(data['queries'])} queries.")

Loaded 10 records, 2 queries.


## Step 1 — Store memories

The agent stores 10 memories: 2 truly useful (the 401 cause and fix), 3 misleading near-matches,
and 5 noise records.

In [2]:
mem = Memory()
record_map = {}   # scenario_id → memory record id

for r in data['records']:
    record = mem.remember(r['content'], kind=r['kind'], metadata=r.get('metadata', {}))
    record_map[r['id']] = record.id

rev_map = {v: k for k, v in record_map.items()}

print('Stored memories:')
for r in data['records']:
    print(f"  [{r['id']}] ({r['kind']}) {r['content'][:65]}")

Stored memories:
  [r1] (observation) GitHub API request failed with 401 because the access token had e
  [r2] (procedure) Refreshing the GitHub token and retrying fixed the 401 error.
  [r3] (observation) GitHub API rate limit caused a temporary 403 response.
  [r4] (observation) Slack webhook failed because the signing secret was missing.
  [r5] (note) Use terse bullet points in status updates.
  [r6] (observation) Database connection failed due to an incorrect host value.
  [r7] (procedure) Check whether the environment variables are loaded before retryin
  [r8] (observation) GitHub pull request fetch worked after widening repository permis
  [r9] (note) Retry with exponential backoff for transient upstream failures.
  [r10] (observation) The CI pipeline failed because the package lockfile was out of da


## Step 2 — Recall before feedback

The agent encounters a GitHub API 401 and asks for useful context.
At this point the system has no learned preference — it relies on keyword matching.

In [3]:
def show_hits(hits, oracle_ids, max_results=5):
    print(f"{'Rank':<5} {'ID':<5} {'Score':<8} {'Useful':<8} Content")
    print('-' * 80)
    for i, h in enumerate(hits[:max_results]):
        sid = rev_map.get(h.id, '?')
        useful = '✓' if sid in oracle_ids else ''
        print(f"{i+1:<5} {sid:<5} {h.score:<8.4f} {useful:<8} {h.content[:55]}")

q1 = next(q for q in data['queries'] if q['id'] == 'q1')
q2 = next(q for q in data['queries'] if q['id'] == 'q2')

print(f"Query: {q1['text']}")
print(f"Oracle-useful: {q1['oracle_useful']}\n")
hits_q1_before = mem.recall(q1['text'], limit=5)
show_hits(hits_q1_before, q1['oracle_useful'])

Query: How should I fix this GitHub API 401 failure?
Oracle-useful: ['r1', 'r2']

Rank  ID    Score    Useful   Content
--------------------------------------------------------------------------------
1     r1    0.4550   ✓        GitHub API request failed with 401 because the access t
2     r2    0.3727   ✓        Refreshing the GitHub token and retrying fixed the 401 
3     r3    0.3700            GitHub API rate limit caused a temporary 403 response.
4     r8    0.2829            GitHub pull request fetch worked after widening reposit
5     r7    0.2829            Check whether the environment variables are loaded befo


In [4]:
print(f"Query: {q2['text']}")
print(f"Oracle-useful: {q2['oracle_useful']}\n")
hits_q2_before = mem.recall(q2['text'], limit=5)
show_hits(hits_q2_before, q2['oracle_useful'])
print()
print('Note: r3 (rate-limit — misleading) appears at rank 1 for q2.')
print('The system does not yet know that r3 is unhelpful for 401 errors.')

Query: What past context is most useful for a failing authenticated GitHub API call?
Oracle-useful: ['r1', 'r2', 'r7']

Rank  ID    Score    Useful   Content
--------------------------------------------------------------------------------
1     r3    0.4588            GitHub API rate limit caused a temporary 403 response.
2     r7    0.4575   ✓        Check whether the environment variables are loaded befo
3     r1    0.4575   ✓        GitHub API request failed with 401 because the access t
4     r2    0.4056   ✓        Refreshing the GitHub token and retrying fixed the 401 
5     r8    0.4043            GitHub pull request fetch worked after widening reposit

Note: r3 (rate-limit — misleading) appears at rank 1 for q2.
The system does not yet know that r3 is unhelpful for 401 errors.


## Step 3 — Apply feedback

The agent used the retrieved context and now reports what was helpful:
- r1 and r2 (cause and fix) — **helpful**
- r3 (rate limit) — **not helpful** (different error code, different cause)

In [5]:
fp = data['feedback_positive']
fn = data['feedback_negative']

pos_memory_ids = [record_map[sid] for sid in fp['target_ids'] if sid in record_map]
neg_memory_ids = [record_map[sid] for sid in fn['target_ids'] if sid in record_map]

mem.feedback(pos_memory_ids, helped=fp['helped'],  reason=fp['reason'])
mem.feedback(neg_memory_ids, helped=fn['helped'],  reason=fn['reason'])

print(f"Positive feedback ({fp['helped']}) → {fp['target_ids']}")
print(f"  Reason: {fp['reason']}")
print()
print(f"Negative feedback ({fn['helped']}) → {fn['target_ids']}")
print(f"  Reason: {fn['reason']}")

Positive feedback (1.0) → ['r1', 'r2']
  Reason: These captured the true cause and the successful fix.

Negative feedback (0.0) → ['r3']
  Reason: Rate limiting looked similar but was not the cause of the 401.


## Step 4 — Recall after feedback

The system has now learned from the feedback. Re-running the same queries should show
the useful records ranked higher and the misleading r3 pushed down.

In [6]:
print(f"Query: {q1['text']}\n")
hits_q1_after = mem.recall(q1['text'], limit=5)
show_hits(hits_q1_after, q1['oracle_useful'])

Query: How should I fix this GitHub API 401 failure?

Rank  ID    Score    Useful   Content
--------------------------------------------------------------------------------
1     r1    0.6406   ✓        GitHub API request failed with 401 because the access t
2     r2    0.5501   ✓        Refreshing the GitHub token and retrying fixed the 401 
3     r3    0.5197            GitHub API rate limit caused a temporary 403 response.
4     r8    0.4496            GitHub pull request fetch worked after widening reposit
5     r7    0.4496            Check whether the environment variables are loaded befo


In [7]:
print(f"Query: {q2['text']}\n")
hits_q2_after = mem.recall(q2['text'], limit=5)
show_hits(hits_q2_after, q2['oracle_useful'])

Query: What past context is most useful for a failing authenticated GitHub API call?

Rank  ID    Score    Useful   Content
--------------------------------------------------------------------------------
1     r1    0.4758   ✓        GitHub API request failed with 401 because the access t
2     r7    0.4719   ✓        Check whether the environment variables are loaded befo
3     r3    0.4626            GitHub API rate limit caused a temporary 403 response.
4     r8    0.4187            GitHub pull request fetch worked after widening reposit
5     r2    0.4187   ✓        Refreshing the GitHub token and retrying fixed the 401 


## Step 5 — Rank delta analysis

The canonical proof moment: how much did each record's rank change?

In [8]:
def rank_of(hits, scenario_id):
    for i, h in enumerate(hits):
        if rev_map.get(h.id) == scenario_id:
            return i
    return len(hits)

print(f"{'ID':<5} {'Label':<15} {'Before':>8} {'After':>8} {'Delta':>8}")
print('-' * 48)

oracle_q2 = set(q2['oracle_useful'])
for sid in ['r1', 'r2', 'r3', 'r7']:
    before = rank_of(hits_q2_before, sid)
    after  = rank_of(hits_q2_after, sid)
    delta  = before - after
    label  = '(useful)' if sid in oracle_q2 else '(misleading)'
    arrow  = '▲' if delta > 0 else ('▼' if delta < 0 else '─')
    print(f"{sid:<5} {label:<15} {before+1:>8} {after+1:>8} {arrow} {abs(delta):>5}")

ID    Label             Before    After    Delta
------------------------------------------------
r1    (useful)               3        1 ▲     2
r2    (useful)               4        5 ▼     1
r3    (misleading)           1        3 ▼     2
r7    (useful)               2        2 ─     0


## Step 6 — Compare to keyword baseline

The misleading r3 has a high keyword score for q2 (shares 'GitHub', 'API', 'failure').
Keyword-only retrieval cannot distinguish it from the useful records.
Neural Ledger can, because it learned from feedback.

In [9]:
from benchmarks.harness import KeywordBaseline, load_dataset

ds = load_dataset()
kw = KeywordBaseline(ds)
q2_obj = next(q for q in ds.queries if q.id == 'q2')

kw_result = kw.retrieve(q2_obj)
nl_result_before = [rev_map.get(h.id) for h in hits_q2_before]
nl_result_after  = [rev_map.get(h.id) for h in hits_q2_after]

print(f"{'Rank':<5} {'Keyword':<8} {'NL before':<12} {'NL after':<10}")
print('-' * 40)
for i in range(5):
    kw_id = kw_result.ranked_ids[i] if i < len(kw_result.ranked_ids) else '-'
    nl_b  = nl_result_before[i]     if i < len(nl_result_before)     else '-'
    nl_a  = nl_result_after[i]      if i < len(nl_result_after)      else '-'
    print(f"{i+1:<5} {kw_id:<8} {str(nl_b):<12} {str(nl_a):<10}")

print()
print(f"Mean useful rank — keyword  : {kw_result.mean_useful_rank():.2f}")

def mean_rank(hits, oracle):
    ranks = [next((i for i, h in enumerate(hits) if rev_map.get(h.id) == sid), len(hits))
             for sid in oracle]
    return sum(ranks) / len(ranks) if ranks else 0

print(f"Mean useful rank — NL before: {mean_rank(hits_q2_before, q2['oracle_useful']):.2f}")
print(f"Mean useful rank — NL after : {mean_rank(hits_q2_after,  q2['oracle_useful']):.2f}")

ModuleNotFoundError: No module named 'benchmarks'

## Step 7 — Telemetry

In [ ]:
import pprint
pprint.pprint(mem.metrics())

## Summary

| Signal | Before feedback | After feedback |
|---|---|---|
| r1 rank (q2) | 3 | **1** |
| r3 rank (q2) | 1 | **3** |
| Mean useful rank (q2) | 2.00 | **1.67** |
| Keyword mean useful rank | 2.00 | 2.00 (unchanged) |

**Neural Ledger learned what past experience is useful — keyword retrieval cannot.**